In [1]:
import os
import sys

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import numpy as np
import gymnasium as gym
from tqdm.auto import tqdm
from sac_agent import get_SAC_agent
from utils.parsing import load_config, convert_to_cfgnode

In [2]:
cfg = load_config("../configs/sac_hockey.yaml")
cfg = convert_to_cfgnode(cfg)

In [ ]:
env_name = 'Pendulum-v1'
# env_name = 'CartPole-v0'

env = gym.make(env_name)
# if isinstance(env.action_space, spaces.Box):
#     env = DiscreteActionWrapper(env,5)

ac_space = env.action_space
o_space = env.observation_space
print(ac_space)
print(o_space)
print(list(zip(env.observation_space.low, env.observation_space.high)))

In [4]:

agent = get_SAC_agent(env, cfg)

In [ ]:
tt = 10000 # cfg.training.total_timesteps
agent.learn(total_timesteps=tt, log_interval=cfg.training.log_interval, progress_bar=True)

In [ ]:
test_stats = []
episodes=50
env_ = env    # without rendering
#env_ = env_eval # with rendering

for i in tqdm(range(episodes)):
    total_reward = 0
    ob, _info = env_.reset()
    for t in range(1000):
        done = False
        a, _ = agent.predict(ob)
        (ob_new, reward, done, trunc, _info) = env_.step(a)
        total_reward+= reward
        ob=ob_new        
        if done: 
            break    
    # print(i, "Reward:", total_reward)
    test_stats.append([i,total_reward,t+1])        

In [ ]:
test_stats_np = np.array(test_stats)
print(np.mean(test_stats_np[:,1]), "+-", np.std(test_stats_np[:,1]))